# 🌿 Jute Pest Classification — EfficientNetB0 Transfer Learning

> **Task:** 17-class image classification of jute pests using Transfer Learning on EfficientNetB0 (pretrained on ImageNet).  
> **Target:** ≥ 95% test accuracy with F1-score reporting and ONNX export.

---
### ⚡ Pipeline
1. Install & imports
2. Download dataset via KaggleHub
3. Explore dataset structure
4. Build data pipelines with augmentation
5. Build EfficientNetB0 model (Phase 1: frozen backbone)
6. Phase 2: Fine-tune top layers
7. Evaluate — Accuracy, F1, Confusion Matrix
8. Export to ONNX & verify

> 💡 **Run on GPU** — Runtime → Change runtime type → T4 GPU (free on Colab)

## ① Install Dependencies

In [ ]:
!pip install -q kagglehub tf2onnx onnxruntime scikit-learn matplotlib seaborn

## ② Imports & GPU Check

In [ ]:
import os, pathlib, shutil, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
warnings.filterwarnings('ignore')

# Confirm GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {'✅ ' + gpus[0].name if gpus else '❌ No GPU — switch runtime to T4!'}")

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Constants
IMG_SIZE    = 224        # EfficientNetB0 native size
BATCH_SIZE  = 32
NUM_CLASSES = 17
AUTOTUNE    = tf.data.AUTOTUNE

## ③ Download Dataset from Kaggle

In [ ]:
import kagglehub

path = kagglehub.dataset_download("zsinghrahulk/jute-pest-images")
print("Dataset downloaded to:", path)

# Walk the directory tree to understand structure
print("\n📁 Directory structure (top 3 levels):")
for root, dirs, files in os.walk(path):
    depth = root.replace(path, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

## ④ Explore Dataset Structure

In [ ]:
# Auto-detect train/val/test split folders
data_root = pathlib.Path(path)

# Find the folder containing class subfolders
def find_split_dir(root, name_hints):
    for hint in name_hints:
        for p in root.rglob(hint):
            if p.is_dir():
                return p
    return None

train_dir = find_split_dir(data_root, ['train', 'Train', 'training'])
val_dir   = find_split_dir(data_root, ['val', 'Val', 'valid', 'validation'])
test_dir  = find_split_dir(data_root, ['test', 'Test', 'testing'])

print(f"Train dir : {train_dir}")
print(f"Val dir   : {val_dir}")
print(f"Test dir  : {test_dir}")

# If no pre-split, we'll split manually below (handled automatically)
if train_dir is None:
    print("⚠️  No train/val/test split found — will create one from root.")
    # Find the folder with class subfolders
    for p in sorted(data_root.rglob('*')):
        if p.is_dir() and any(c.is_dir() for c in p.iterdir()):
            train_dir = p
            break
    print(f"Using root: {train_dir}")

# Count classes and images
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
print(f"\n📊 Classes found ({len(classes)}):")
img_counts = {}
for cls in classes:
    imgs = list((train_dir / cls).glob('*.*'))
    img_counts[cls] = len(imgs)
    print(f"  {cls:35s}: {len(imgs)} images")
print(f"\nTotal train images: {sum(img_counts.values())}")

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
axes = axes.ravel()
fig.suptitle('Sample Images per Jute Pest Class', fontsize=14, fontweight='bold')

for i, cls in enumerate(classes[:17]):
    imgs = list((train_dir / cls).glob('*.*'))
    if not imgs:
        continue
    img = plt.imread(str(imgs[0]))
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=7, fontweight='bold')
    axes[i].axis('off')

# Hide unused axes
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

# Class distribution bar chart
plt.figure(figsize=(14, 4))
plt.bar(img_counts.keys(), img_counts.values(), color='steelblue', edgecolor='black')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('Images per Class (Train)', fontsize=13, fontweight='bold')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## ⑤ Build tf.data Pipelines

In [ ]:
# ── Data augmentation (training only) ──────────────────────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

# ── Build datasets ──────────────────────────────────────────────────────────
def make_dataset(directory, training=False, val_split=None, subset=None):
    kwargs = dict(
        directory   = str(directory),
        image_size  = (IMG_SIZE, IMG_SIZE),
        batch_size  = BATCH_SIZE,
        label_mode  = 'categorical',
        seed        = 42,
    )
    if val_split:
        kwargs['validation_split'] = val_split
        kwargs['subset'] = subset
    ds = keras.utils.image_dataset_from_directory(**kwargs)
    class_names = ds.class_names
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(AUTOTUNE)
    return ds, class_names

# Case 1: pre-split dataset (train / val / test folders exist)
if val_dir and test_dir:
    train_ds, class_names = make_dataset(train_dir, training=True)
    val_ds,   _           = make_dataset(val_dir,   training=False)
    test_ds,  _           = make_dataset(test_dir,  training=False)
    print("✅ Using pre-split train / val / test folders")

# Case 2: single folder — split 70/15/15
else:
    # First create train+val (85%) vs test (15%)
    trainval_ds, class_names = make_dataset(train_dir, training=False,
                                            val_split=0.15, subset='training')
    test_ds, _ = make_dataset(train_dir, training=False,
                              val_split=0.15, subset='validation')
    # Split trainval into 82% train / 18% val  ≈  70/15 of total
    n = tf.data.experimental.cardinality(trainval_ds).numpy()
    train_size = int(0.82 * n)
    train_ds = trainval_ds.take(train_size)
    val_ds   = trainval_ds.skip(train_size)
    # Apply augmentation to train only
    train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                            num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    print("✅ Auto-split: 70% train / 15% val / 15% test")

print(f"\nClass names ({len(class_names)}): {class_names}")
print(f"Train batches : {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Val batches   : {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Test batches  : {tf.data.experimental.cardinality(test_ds).numpy()}")

## ⑥ Build Model — EfficientNetB0 + Custom Head

**Strategy:**  
- **Phase 1 (warm-up):** Freeze EfficientNetB0 backbone, train only the new head for 10 epochs → fast convergence.  
- **Phase 2 (fine-tune):** Unfreeze top 50 layers, train end-to-end with a lower LR for 20 epochs → pushes to 95%+.

In [ ]:
def build_efficientnet_model(num_classes, img_size=224):
    # Preprocessing layer (EfficientNet expects [0,255] and scales internally)
    inputs = keras.Input(shape=(img_size, img_size, 3), name='image_input')

    # Backbone — pretrained on ImageNet, no top
    backbone = EfficientNetB0(
        include_top  = False,
        weights      = 'imagenet',
        input_tensor = inputs
    )
    backbone.trainable = False  # Phase 1: frozen

    # Custom classification head
    x = backbone.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu', name='fc2')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs, outputs, name='JutePest_EfficientNetB0')
    return model, backbone

model, backbone = build_efficientnet_model(NUM_CLASSES)

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

# Summary (just the head)
print(f"Backbone layers  : {len(backbone.layers)}")
print(f"Trainable params : {model.count_params():,}")
model.summary(show_trainable=True)

## ⑦ Phase 1 — Warm-Up Training (Frozen Backbone)

In [ ]:
PHASE1_EPOCHS = 10

callbacks_p1 = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint('best_phase1.keras', save_best_only=True,
                                    monitor='val_accuracy', verbose=0)
]

print("=" * 55)
print("  PHASE 1 — Head-only warm-up (backbone frozen)")
print("=" * 55)

history1 = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = PHASE1_EPOCHS,
    callbacks       = callbacks_p1,
    verbose         = 1
)

p1_acc = max(history1.history['val_accuracy'])
print(f"\n✅ Phase 1 best val accuracy: {p1_acc*100:.2f}%")

## ⑧ Phase 2 — Fine-Tune Top Backbone Layers

In [ ]:
PHASE2_EPOCHS   = 25
UNFREEZE_FROM   = -50   # Unfreeze last 50 layers of backbone

# Unfreeze selected layers
backbone.trainable = True
for layer in backbone.layers[:UNFREEZE_FROM]:
    layer.trainable = False

trainable_now = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers after unfreeze: {trainable_now}")

# Recompile with lower LR
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-4),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

callbacks_p2 = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                      patience=3, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint('best_model.keras', save_best_only=True,
                                    monitor='val_accuracy', verbose=0)
]

print("\n" + "=" * 55)
print("  PHASE 2 — Fine-tuning top backbone layers")
print("=" * 55)

history2 = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = PHASE2_EPOCHS,
    callbacks       = callbacks_p2,
    verbose         = 1
)

p2_acc = max(history2.history['val_accuracy'])
print(f"\n✅ Phase 2 best val accuracy: {p2_acc*100:.2f}%")

In [ ]:
# Combine history from both phases
acc  = history1.history['accuracy']     + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
p1_end = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History (Phase 1 → Phase 2)', fontsize=14, fontweight='bold')

for ax, train_m, val_m, title, ylabel in zip(
    axes,
    [acc, loss], [val_acc, val_loss],
    ['Accuracy', 'Loss'], ['Accuracy', 'Loss']
):
    ax.plot(train_m, label=f'Train {ylabel}', color='royalblue')
    ax.plot(val_m,   label=f'Val {ylabel}',   color='orange', linestyle='--')
    ax.axvline(p1_end - 1, color='red', linestyle=':', linewidth=1.5, label='Fine-tune start')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.4)

plt.tight_layout()
plt.show()

## ⑨ Evaluation — Accuracy, F1, Confusion Matrix

In [ ]:
# Collect predictions on the test set
print("Running inference on test set...")
y_true_all, y_pred_all = [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred_all.extend(np.argmax(preds, axis=1))
    y_true_all.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true_all)
y_pred = np.array(y_pred_all)

# ── Core Metrics ──────────────────────────────────────────────────────────
test_acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_weighted = f1_score(y_true, y_pred, average='weighted')

print(f"\n{'='*50}")
print(f"  Test Accuracy     : {test_acc*100:.2f}%")
print(f"  F1-Score (Macro)  : {f1_macro:.4f}")
print(f"  F1-Score (Weighted): {f1_weighted:.4f}")
print(f"{'='*50}\n")

if test_acc >= 0.95:
    print("🎯 Target of ≥95% accuracy ACHIEVED!")
else:
    print(f"⚠️  Accuracy is {test_acc*100:.2f}% — try adding more epochs in Phase 2")

print("\n📋 Per-class Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, ax=ax
)
ax.set_title(f'Confusion Matrix — Test Accuracy: {test_acc*100:.2f}%',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Class', fontsize=11)
ax.set_ylabel('True Class', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

# Per-class F1 bar chart
per_class_f1 = f1_score(y_true, y_pred, average=None)
colors = ['#2ecc71' if s >= 0.95 else '#e74c3c' for s in per_class_f1]

plt.figure(figsize=(14, 5))
bars = plt.bar(class_names, per_class_f1, color=colors, edgecolor='black')
plt.axhline(0.95, color='navy', linestyle='--', linewidth=1.5, label='95% threshold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.ylabel('F1-Score')
plt.title('Per-Class F1-Score', fontsize=13, fontweight='bold')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(axis='y', alpha=0.4)
for bar, score in zip(bars, per_class_f1):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.2f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()

## ⑩ Export to ONNX & Verify Accuracy

In [ ]:
import tf2onnx
import onnx

ONNX_PATH = "jute_pest_efficientnet.onnx"

# Convert Keras model → ONNX
input_sig = [tf.TensorSpec(
    shape  = (None, IMG_SIZE, IMG_SIZE, 3),
    dtype  = tf.float32,
    name   = 'image_input'
)]

print("Converting to ONNX (opset 13)...")
onnx_model, _ = tf2onnx.convert.from_keras(
    model,
    input_signature = input_sig,
    opset           = 13
)

onnx.save(onnx_model, ONNX_PATH)
file_size_mb = os.path.getsize(ONNX_PATH) / 1e6
print(f"✅ ONNX model saved → {ONNX_PATH}  ({file_size_mb:.1f} MB)")

# Validate the graph
onnx.checker.check_model(ONNX_PATH)
print("✅ ONNX graph validation passed")

In [ ]:
import onnxruntime as ort

# Create runtime session
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
inp_name  = sess.get_inputs()[0].name
out_name  = sess.get_outputs()[0].name

print(f"ONNX Input  : '{inp_name}'  shape={sess.get_inputs()[0].shape}")
print(f"ONNX Output : '{out_name}' shape={sess.get_outputs()[0].shape}")

# Run inference on test set and measure accuracy
print("\nRunning ONNX inference on test set...")
onnx_preds = []

for images, labels in test_ds:
    batch = images.numpy().astype(np.float32)
    probs = sess.run([out_name], {inp_name: batch})[0]
    onnx_preds.extend(np.argmax(probs, axis=1))

onnx_preds  = np.array(onnx_preds)
onnx_acc    = accuracy_score(y_true, onnx_preds)
onnx_f1     = f1_score(y_true, onnx_preds, average='macro')

print(f"\n{'='*50}")
print(f"  Keras  Test Accuracy  : {test_acc*100:.2f}%")
print(f"  ONNX   Test Accuracy  : {onnx_acc*100:.2f}%")
print(f"  Keras  F1 (Macro)     : {f1_macro:.4f}")
print(f"  ONNX   F1 (Macro)     : {onnx_f1:.4f}")
print(f"  Prediction match rate : {np.mean(y_pred == onnx_preds)*100:.2f}%")
print(f"{'='*50}")

if abs(test_acc - onnx_acc) < 0.005:
    print("\n✅ ONNX model matches Keras — export verified!")
else:
    print("\n⚠️  Small numerical difference (normal for float32 conversion)")

In [ ]:
# Download the ONNX model
from google.colab import files
files.download(ONNX_PATH)
print(f"⬇️  '{ONNX_PATH}' downloaded successfully!")

---
## ✅ Summary

| Item | Detail |
|---|---|
| **Dataset** | Jute Pest Images — 17 classes, ~380 images/class |
| **Model** | EfficientNetB0 (ImageNet pretrained) + custom head |
| **Strategy** | 2-phase: warm-up (10 ep) → fine-tune top-50 layers (25 ep) |
| **Augmentation** | Flip, Rotation, Zoom, Brightness, Contrast |
| **Metrics** | Accuracy + Macro F1 + Per-class F1 + Confusion Matrix |
| **Export** | ONNX opset-13 via `tf2onnx`, verified with `onnxruntime` |
| **Output file** | `jute_pest_efficientnet.onnx` |

### 💡 If accuracy < 95%:
- Increase `PHASE2_EPOCHS` to 40
- Change `UNFREEZE_FROM = -80` to unfreeze more layers
- Try `EfficientNetB3` for a stronger backbone (slightly slower)